In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('dog_food').getOrCreate()
spark

In [5]:
# Read data from CSV file
df = spark.read.csv('dog_food.csv', sep=',', header=True, inferSchema=True,
 nullValue='NA')

In [6]:
#Get number of records
print("The data contain %d records." % df.count())

The data contain 490 records.


In [7]:
#View the first five records
df.show(5)

+---+---+----+---+-------+
|  A|  B|   C|  D|Spoiled|
+---+---+----+---+-------+
|  4|  2|12.0|  3|    1.0|
|  5|  6|12.0|  7|    1.0|
|  6|  2|13.0|  6|    1.0|
|  4|  2|12.0|  1|    1.0|
|  4|  2|12.0|  3|    1.0|
+---+---+----+---+-------+
only showing top 5 rows



In [8]:
# Check column data types
print(df.printSchema())
print(df.dtypes)

root
 |-- A: integer (nullable = true)
 |-- B: integer (nullable = true)
 |-- C: double (nullable = true)
 |-- D: integer (nullable = true)
 |-- Spoiled: double (nullable = true)

None
[('A', 'int'), ('B', 'int'), ('C', 'double'), ('D', 'int'), ('Spoiled', 'double')]


In [9]:
from pyspark.ml.feature import VectorAssembler
# Create an assembler object
assembler = VectorAssembler(inputCols=['A','B','C','D'], outputCol='features')

In [10]:
# Consolidate predictor columns
df_assembled = assembler.transform(df)
# Check the resulting column
df_assembled.select('features', 'Spoiled').show(5, truncate=False)

+------------------+-------+
|features          |Spoiled|
+------------------+-------+
|[4.0,2.0,12.0,3.0]|1.0    |
|[5.0,6.0,12.0,7.0]|1.0    |
|[6.0,2.0,13.0,6.0]|1.0    |
|[4.0,2.0,12.0,1.0]|1.0    |
|[4.0,2.0,12.0,3.0]|1.0    |
+------------------+-------+
only showing top 5 rows



In [11]:
df_assembled.count() 

490

In [12]:
# Train/test split
df_train, df_test = df_assembled.randomSplit([0.8, 0.2], seed=17)
# Check that training set has around 80% of records
training_ratio = df_train.count() / df_assembled.count()
print(training_ratio)

0.7795918367346939


In [17]:
# Build a Decision Tree
from pyspark.ml.classification import DecisionTreeClassifier
# Create a classifier object and fit to the training data
tree = DecisionTreeClassifier(labelCol='Spoiled',featuresCol='features')
tree_model = tree.fit(df_train)

In [19]:
# Create predictions for the testing data and take a look at the predictions
prediction = tree_model.transform(df_test)
prediction.select('Spoiled', 'prediction', 'probability').show(5, False)

+-------+----------+-----------+
|Spoiled|prediction|probability|
+-------+----------+-----------+
|1.0    |0.0       |[0.85,0.15]|
|0.0    |0.0       |[1.0,0.0]  |
|0.0    |0.0       |[1.0,0.0]  |
|0.0    |0.0       |[1.0,0.0]  |
|0.0    |0.0       |[1.0,0.0]  |
+-------+----------+-----------+
only showing top 5 rows



In [21]:
# Evaluate the Decision Tree
prediction.groupBy('Spoiled', 'prediction').count().show()
# Calculate the elements of the confusion matrix
TN = prediction.filter('prediction = 0 AND Spoiled = prediction').count()
TP = prediction.filter('prediction = 1 AND Spoiled = prediction').count() 
FN = prediction.filter('prediction = 0 AND Spoiled = 1').count()
FP = prediction.filter('prediction = 1 AND Spoiled = 0').count()
# Accuracy measures the proportion of correct predictions
accuracy = (TN + TP) / (TN + TP + FN + FP)
print(accuracy)

+-------+----------+-----+
|Spoiled|prediction|count|
+-------+----------+-----+
|    1.0|       1.0|   22|
|    0.0|       1.0|    2|
|    1.0|       0.0|    2|
|    0.0|       0.0|   82|
+-------+----------+-----+

0.9629629629629629


In [22]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
# Calculate precision and recall
precision = TP / (TP + FP)
recall = TP / (TP + FN)
print('precision = {:.2f}\nrecall = {:.2f}'.format(precision, recall))

precision = 0.92
recall = 0.92
